# Mixture of Experts (MoE) on the SOTA Ensemble — CIFAR-10

## Goal

Add a learned **router/gating network** that decides, per input image, how to
weight the two SOTA experts (ResNet18-sota + DenseNet121-sota):

$$\mathbf{p}(x) = \sum_{i} g_i(x)\,\mathbf{p}_i(x), \qquad
g(x) = \operatorname{softmax}\big(\operatorname{Router}(\,[p_1(x), p_2(x)])\,\big)$$

Implemented in two phases:
- **Phase 1 — frozen experts + soft router:** experts are frozen; only the
  small router is trained on the **validation** split. Cheap, fast.
- **Phase 2 — joint fine-tune:** unfreeze the experts' top blocks and train
  router + experts together on the **train** split (heavier; gated by
  `RUN_PHASE2`).

Baselines to beat: soft-voting 96.87%, +TTA 97.12%, +MLP stacking 97.21%.

## 1. Setup & Data

Validation split (train the router), train split (Phase 2), test split
(evaluate). All at 224x224 with the ImageNet eval transform.

In [1]:
import json, sys, os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

_cwd = Path(os.getcwd()).resolve()
PROJECT_ROOT = _cwd if (_cwd / "src").exists() else (_cwd.parent if (_cwd.parent / "src").exists() else _cwd)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.build_model import build_resnet18, build_densenet121, set_parameter_requires_grad
from src.eval.evaluate_model import load_checkpoint, CIFAR10_CLASSES

CAT_IDX, DOG_IDX = 3, 5
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
DATA_ROOT = str(PROJECT_ROOT / "data" / "raw")
SPLIT_FILE = str(PROJECT_ROOT / "data" / "processed" / "cifar10_split_seed42.json")
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

split = json.loads(Path(SPLIT_FILE).read_text())
tform = transforms.Compose([transforms.Resize(224), transforms.ToTensor(),
                            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

def make_loader(train, indices, batch_size=64, transform=tform):
    ds = torchvision.datasets.CIFAR10(DATA_ROOT, train=train, transform=transform)
    if indices is not None:
        ds = Subset(ds, indices)
    return DataLoader(ds, batch_size=batch_size, shuffle=False)

val_loader  = make_loader(True, split["val_indices"])
test_loader = make_loader(False, None)
# Train loader (Phase 2) with light augmentation
train_tform = transforms.Compose([transforms.Resize(224), transforms.RandomHorizontalFlip(),
                                  transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
train_loader = make_loader(True, split["train_indices"], transform=train_tform)
print(f"train: {len(train_loader.dataset)}, val: {len(val_loader.dataset)}, test: {len(test_loader.dataset)}")

Device: cuda
train: 45000, val: 5000, test: 10000


## 2. Load SOTA Experts & Helpers

In [2]:
ckpt = PROJECT_ROOT / "experiments" / "checkpoints"
rn = build_resnet18(num_classes=10, mode="finetune", device=device)
dn = build_densenet121(num_classes=10, mode="finetune", device=device)
rn = load_checkpoint(rn, str(ckpt / "ResNet18-sota_best.pt"), device)
dn = load_checkpoint(dn, str(ckpt / "DenseNet121-sota_best.pt"), device)
rn.eval(); dn.eval()
print("Loaded ResNet18-sota and DenseNet121-sota experts.")

@torch.inference_mode()
def base_features(loader, device, tta=False):
    """Return (p1[N,10], p2[N,10], y[N]) for ResNet & DenseNet experts."""
    p1s, p2s, ys = [], [], []
    for x, y in loader:
        x = x.to(device)
        if tta:
            views = [x, torch.flip(x, dims=[3])]
            a = torch.stack([torch.softmax(rn(v), 1) for v in views]).mean(0)
            b = torch.stack([torch.softmax(dn(v), 1) for v in views]).mean(0)
        else:
            a = torch.softmax(rn(x), 1); b = torch.softmax(dn(x), 1)
        p1s.append(a.cpu()); p2s.append(b.cpu()); ys.append(y)
    return torch.cat(p1s).numpy(), torch.cat(p2s).numpy(), torch.cat(ys).numpy()

def full_metrics(probs, targets):
    preds = np.argmax(probs, axis=1)
    acc = (preds == targets).mean() * 100.0
    cm = confusion_matrix(targets, preds, labels=list(range(10)))
    rep = classification_report(targets, preds, labels=list(range(10)), output_dict=True, zero_division=0)
    return acc, rep["macro avg"]["f1-score"], cm

def isolated_catdog(probs, targets):
    mask = (targets == CAT_IDX) | (targets == DOG_IDX)
    t, p = targets[mask], np.argmax(probs[mask], axis=1)
    acc = (p == t).mean() * 100.0
    cross = int(((t == CAT_IDX) & (p == DOG_IDX)).sum() + ((t == DOG_IDX) & (p == CAT_IDX)).sum())
    return acc, cross

def report(name, probs, targets):
    acc, f1, cm = full_metrics(probs, targets)
    iso, cross = isolated_catdog(probs, targets)
    print(f"{name:34s} acc={acc:6.2f}%  f1={f1:.4f}  isolated={iso:5.2f}%  cross={cross:3d}")
    return acc, f1, iso, cross, cm

c:\document\Study documents\Deeplearning_Course\.venv\Lib\site-packages\torch\nn\modules\module.py:1369: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  return t.to(


Loaded ResNet18-sota and DenseNet121-sota experts.


## 3. Extract base probabilities (val for router, test for eval)

In [3]:
# no-TTA features (Phase 1 primary)
p1_val, p2_val, y_val = base_features(val_loader, device, tta=False)
p1_test, p2_test, y_test = base_features(test_loader, device, tta=False)
# TTA features (for eval comparison)
p1_testT, p2_testT, _ = base_features(test_loader, device, tta=True)
print("val:", p1_val.shape, "test:", p1_test.shape)

# ---- Baselines: fixed soft-voting ----
p_ens = 0.5 * (p1_test + p2_test)
p_ensT = 0.5 * (p1_testT + p2_testT)
print("\n=== Baselines ===")
base_metrics = report("Soft-voting (no TTA)", p_ens, y_test)
report("Soft-voting + TTA", p_ensT, y_test)

val: (5000, 10) test: (10000, 10)

=== Baselines ===
Soft-voting (no TTA)               acc= 96.87%  f1=0.9687  isolated=93.45%  cross= 86
Soft-voting + TTA                  acc= 97.12%  f1=0.9712  isolated=94.00%  cross= 76


(np.float64(97.11999999999999),
 0.9711946395030082,
 np.float64(94.0),
 76,
 array([[982,   1,   4,   1,   0,   0,   0,   1,   7,   4],
        [  0, 980,   0,   0,   0,   0,   0,   1,   1,  18],
        [  4,   0, 966,  11,   5,   3,   8,   1,   2,   0],
        [  1,   1,   4, 937,  10,  33,   5,   5,   3,   1],
        [  1,   0,   2,   8, 977,   2,   2,   8,   0,   0],
        [  0,   0,   3,  43,   7, 943,   1,   3,   0,   0],
        [  2,   0,   4,   5,   1,   1, 987,   0,   0,   0],
        [  1,   0,   4,   2,   5,   5,   0, 983,   0,   0],
        [  7,   4,   1,   0,   0,   0,   0,   0, 981,   7],
        [  2,  19,   0,   0,   0,   0,   0,   0,   3, 976]]))

## 4. PHASE 1 — Frozen experts + soft router

Only the small router is trained, on the **validation** split (leakage-free
w.r.t. test). A load-balancing loss discourages router collapse.

In [4]:
class RouterMLP(nn.Module):
    def __init__(self, in_dim=20, hidden=64, n_experts=2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(inplace=True),
                                 nn.Linear(hidden, n_experts))
    def forward(self, x):
        return self.net(x)

def load_balance_loss(gates, n_experts=2):
    frac = gates.mean(0)          # empirical fraction of inputs routed to each expert
    avg_gate = gates.mean(0)
    return n_experts * (frac * avg_gate).sum()

def train_router_soft(p1, p2, y, p1_val, p2_val, y_val, device, epochs=40,
                      lr=1e-3, alpha=0.1, gate_T=2.0, ent_w=0.1, seed=0):
    """Frozen-expert soft router with gate temperature + entropy regularizer.

    gate_T > 1 softens the gate (avoids near-top-1 collapse); ent_w penalises
    peaked (low-entropy) gates; alpha weights the load-balancing loss.
    """
    torch.manual_seed(seed)
    router = RouterMLP(in_dim=20).to(device)
    opt = torch.optim.Adam(router.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    P1 = torch.tensor(p1, dtype=torch.float32); P2 = torch.tensor(p2, dtype=torch.float32)
    Y = torch.tensor(y, dtype=torch.long)
    V1 = torch.tensor(p1_val, dtype=torch.float32).to(device)
    V2 = torch.tensor(p2_val, dtype=torch.float32).to(device)
    VY = torch.tensor(y_val, dtype=torch.long).to(device)
    best = 0.0
    for ep in range(epochs):
        idx = torch.randperm(len(P1), generator=torch.Generator().manual_seed(seed + ep))
        router.train()
        for b in range(0, len(P1), 256):
            bi = idx[b:b + 256]
            z = router(torch.cat([P1[bi], P2[bi]], 1).to(device)) / gate_T
            g = torch.softmax(z, 1)
            pmix = g[:, 0:1] * P1[bi].to(device) + g[:, 1:2] * P2[bi].to(device)
            ent = -(g * (g + 1e-9).log()).sum(1).mean()
            loss = crit(pmix, Y[bi].to(device)) + alpha * load_balance_loss(g) - ent_w * ent
            opt.zero_grad(); loss.backward(); opt.step()
        router.eval()
        with torch.no_grad():
            gv = torch.softmax(router(torch.cat([V1, V2], 1)) / gate_T, 1)
            pmixv = (gv[:, 0:1] * V1 + gv[:, 1:2] * V2)
            va = float((pmixv.argmax(1).cpu().numpy() == y_val).mean() * 100.0)
        if va > best: best = va
    return router, best

# Train the router on validation features (frozen experts).
router1, val_acc1 = train_router_soft(p1_val, p2_val, y_val, p1_val, p2_val, y_val,
                                      device, epochs=40, seed=0)
print(f"Phase 1 router — val acc: {val_acc1:.2f}%")

def apply_router(router, p1, p2, device):
    with torch.no_grad():
        g = torch.softmax(router(torch.tensor(np.concatenate([p1, p2], 1), dtype=torch.float32).to(device)), 1)
        pmix = (g[:, 0:1].cpu().numpy() * p1) + (g[:, 1:2].cpu().numpy() * p2)
    return pmix, g.cpu().numpy()

# Evaluate Phase 1 on test (no TTA)
p_moe, g_test = apply_router(router1, p1_test, p2_test, device)
print("\n=== Phase 1: frozen experts + soft router (test) ===")
moe1 = report("Ensemble + MoE router (no TTA)", p_moe, y_test)

# With TTA features
p_moeT, g_testT = apply_router(router1, p1_testT, p2_testT, device)
moe1T = report("Ensemble + MoE router + TTA", p_moeT, y_test)

# Routing stats (per class)
print("\nRouting (mean gate for E1=ResNet) per class:")
for c in range(10):
    m = y_test == c
    print(f"  {CIFAR10_CLASSES[c]:11s} g_E1={g_test[m, 0].mean():.3f}  g_E2={g_test[m, 1].mean():.3f}")

Phase 1 router — val acc: 97.00%

=== Phase 1: frozen experts + soft router (test) ===
Ensemble + MoE router (no TTA)     acc= 96.76%  f1=0.9676  isolated=93.05%  cross= 90
Ensemble + MoE router + TTA        acc= 97.04%  f1=0.9704  isolated=93.60%  cross= 83

Routing (mean gate for E1=ResNet) per class:
  airplane    g_E1=0.538  g_E2=0.462
  automobile  g_E1=0.478  g_E2=0.522
  bird        g_E1=0.468  g_E2=0.532
  cat         g_E1=0.442  g_E2=0.558
  deer        g_E1=0.443  g_E2=0.557
  dog         g_E1=0.466  g_E2=0.534
  frog        g_E1=0.403  g_E2=0.597
  horse       g_E1=0.488  g_E2=0.512
  ship        g_E1=0.483  g_E2=0.517
  truck       g_E1=0.537  g_E2=0.463


## 5. PHASE 2 — Joint fine-tune (router + unfrozen expert top blocks)

Heavier and riskier: the experts are already strong, and re-training their top
blocks can degrade them (we saw this in earlier feature-level experiments).
This phase is **gated** by `RUN_PHASE2` (default off) so you control when to
spend the compute. When enabled it:
1. Unfreezes `layer4+fc` (ResNet) and `denseblock4+norm5+classifier` (DenseNet).
2. Trains router + those blocks together on the train split (LLRD),
   with the load-balancing loss.

In [5]:
RUN_PHASE2 = False      # set to True to run Phase 2 (heavy)
EPOCHS_PHASE2 = 2        # joint fine-tune epochs
PHASE2_SUBSET = 0        # 0 = full train; >0 = use that many train samples (smoke)

if RUN_PHASE2:
    if PHASE2_SUBSET > 0:
        sub = list(range(PHASE2_SUBSET))
        train_loader = DataLoader(Subset(train_loader.dataset, sub), batch_size=64, shuffle=True)
    print(f"Running Phase 2: {len(train_loader.dataset)} train samples, {EPOCHS_PHASE2} epochs")

    # Unfreeze expert top blocks
    set_parameter_requires_grad(rn, False); set_parameter_requires_grad(rn.layer4, True); set_parameter_requires_grad(rn.fc, True)
    set_parameter_requires_grad(dn, False); set_parameter_requires_grad(dn.features.denseblock4, True)
    set_parameter_requires_grad(dn.features.norm5, True); set_parameter_requires_grad(dn.classifier, True)

    router2 = RouterMLP(in_dim=20).to(device)
    params = [
        {"params": list(rn.layer4.parameters()) + list(rn.fc.parameters()), "lr": 1e-4},
        {"params": list(dn.features.denseblock4.parameters()) + list(dn.features.norm5.parameters()) + list(dn.classifier.parameters()), "lr": 1e-4},
        {"params": router2.parameters(), "lr": 1e-3},
    ]
    opt = torch.optim.Adam(params, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()

    @torch.inference_mode()
    def gated_eval(tta=False):
        p1t, p2t, yt = base_features(test_loader, device, tta=tta)
        return apply_router(router2, p1t, p2t, device)[0], yt

    for ep in range(EPOCHS_PHASE2):
        rn.train(); dn.train(); router2.train()
        for x, yb in train_loader:
            x, yb = x.to(device), yb.to(device)
            p1 = torch.softmax(rn(x), 1); p2 = torch.softmax(dn(x), 1)
            g = torch.softmax(router2(torch.cat([p1, p2], 1)), 1)
            pmix = g[:, 0:1] * p1 + g[:, 1:2] * p2
            loss = crit(pmix, yb) + 0.01 * load_balance_loss(g)
            opt.zero_grad(); loss.backward(); opt.step()
        rn.eval(); dn.eval(); router2.eval()
        pv, yv = gated_eval(tta=False)
        acc = (pv.argmax(1) == yv).mean() * 100.0
        print(f"  Phase 2 epoch {ep+1}/{EPOCHS_PHASE2} — test acc {acc:.2f}%")

    rn.eval(); dn.eval(); router2.eval()
    p2m, _ = gated_eval(tta=False)
    p2mT, _ = gated_eval(tta=True)
    print("\n=== Phase 2: joint fine-tune (test) ===")
    report("Phase 2 MoE (no TTA)", p2m, y_test)
    report("Phase 2 MoE + TTA", p2mT, y_test)
else:
    print("Phase 2 skipped (RUN_PHASE2 = False). Set it to True to run joint fine-tuning.")

Phase 2 skipped (RUN_PHASE2 = False). Set it to True to run joint fine-tuning.


## 6. Summary

Compare against baselines. Note the Phase 1 router is **learned per-input
soft-voting**; it should be at least as good as the fixed 0.5/0.5 and the MLP
stacking, and it composes with TTA.

In [6]:
print(f"{'Method':34s} {'acc':>7s} {'isolated':>9s} {'cross':>6s}")
print(f"{'Soft-voting (no TTA)':34s} {base_metrics[0]:6.2f}% {base_metrics[2]:8.2f}% {base_metrics[3]:>5d}")
print(f"{'Ensemble + MoE router (no TTA)':34s} {moe1[0]:6.2f}% {moe1[2]:8.2f}% {moe1[3]:>5d}")
print(f"{'Soft-voting + TTA':34s} {base_metrics[0]:6.2f}% {base_metrics[2]:8.2f}% {base_metrics[3]:>5d}  (ref 97.12)")
print(f"{'Ensemble + MoE router + TTA':34s} {moe1T[0]:6.2f}% {moe1T[2]:8.2f}% {moe1T[3]:>5d}")
if 'p2m' in dir():
    print(f"{'Phase 2 MoE + TTA':34s} {report('Phase 2 MoE + TTA', p2mT, y_test)[0]:6.2f}%")
print("\nDone. See agents/progress/MOE_STATUS.md, agents/experiments/MOE_EXPERIMENT.md, agents/phases/MOE.md for the tracking docs.")

Method                                 acc  isolated  cross
Soft-voting (no TTA)                96.87%    93.45%    86
Ensemble + MoE router (no TTA)      96.76%    93.05%    90
Soft-voting + TTA                   96.87%    93.45%    86  (ref 97.12)
Ensemble + MoE router + TTA         97.04%    93.60%    83

Done. See agents/progress/MOE_STATUS.md, agents/experiments/MOE_EXPERIMENT.md, agents/phases/MOE.md for the tracking docs.
